# Generate data tables for paper

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import *
import os
from shared.utils import load_snapshot, run_model
from energyscope.models import Model

In [ ]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL licence file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [ ]:
def format_end_use_category(category):
    category = category.replace('_', ' ')
    category = category.capitalize()
    
    category = category.replace('ehv', 'EHV')
    category = category.replace('hv', 'HV')
    category = category.replace('mv', 'MV')
    category = category.replace('lv', 'LV')

    category = category.replace('ehp', 'EHP')
    category = category.replace('hp', 'HP')
    category = category.replace('mp', 'MP')
    category = category.replace('sp', 'SP')
    
    category = category.replace('eld', 'ELD')
    category = category.replace('ld', 'LD')
    category = category.replace('md', 'MD')
    category = category.replace('sd', 'SD')
    
    category = category.replace('t sc', 'T SC')
    category = category.replace('t sh', 'T SH')
    category = category.replace('t hw', 'T HW')
    category = category.replace('high t', 'high T')
    
    category = category.replace(' export', '')

    category = category.replace('H2', 'Non-energetic H2')
    
    return category

In [ ]:
def physical_unit_end_use_category(category, per_cap=False):
    if per_cap:
        if 'Electricity' in category or 'Lighting' in category:
            return 'kWh/cap'
        elif 'Mobility freight' in category:
            return 'tkm/cap'
        elif 'Mobility passenger' in category:
            return 'pkm/cap'
        elif 'Heat' in category:
            return 'kWh/cap'
        elif 'H2' in category:
            return 'kWh/cap'

    else:
        if 'Electricity' in category or 'Lighting' in category:
            return 'GWh'
        elif 'Mobility freight' in category:
            return 'Mtkm'
        elif 'Mobility passenger' in category:
            return 'Mpkm'
        elif 'Heat' in category:
            return 'GWh'
        elif 'H2' in category:
            return 'GWh'

In [ ]:
def create_demand_table(year, per_cap=False):
    model = load_snapshot(year=year, scenario=False)
    if year == 2023:
        N_capita = N_capita_2023
    elif year == 2050:
        model += Model([('dat', '../02_AMPL_files/data/2050/QC_scenarios.dat')])
        N_capita = N_capita_2050
    else:
        raise ValueError("Year must be either 2023 or 2050.")

    results = run_model(model)
    end_uses_demand_year = results.parameters['end_uses_demand_year'].reset_index().rename(columns={'index0': 'End-use category', 'index1': 'Sector', 'end_uses_demand_year': 'Amount'})
    end_uses_demand_year = end_uses_demand_year[end_uses_demand_year['Amount'] != 0]
    end_uses_demand_year['Sector'] = end_uses_demand_year['Sector'].str.capitalize()
    end_uses_demand_year['End-use category'] = end_uses_demand_year['End-use category'].apply(format_end_use_category)
    if per_cap:
        end_uses_demand_year['Amount'] /= N_capita  # Convert to per capita
        end_uses_demand_year['Amount'] *= 1e6  # Convert to kWh, pkm, tkm
    end_uses_demand_year['Unit'] = end_uses_demand_year['End-use category'].apply(lambda row: physical_unit_end_use_category(category=row, per_cap=per_cap))
    end_uses_demand_year_pivot = end_uses_demand_year.pivot_table(index=['End-use category', 'Unit'], columns='Sector', values='Amount').reset_index()
    end_uses_demand_year_pivot.fillna(0, inplace=True)
    
    return end_uses_demand_year_pivot

## Demands in 2023

In [ ]:
end_uses_demand_year_2023 = create_demand_table(2023)

## Demands in 2050

In [ ]:
end_uses_demand_year_2050 = create_demand_table(2050)

## Combine the two in one table

In [ ]:
end_uses_demand_year_2023.columns = ['End-use category', 'Unit'] + [f'{col} 2023' for col in end_uses_demand_year_2023.columns[2:]]
end_uses_demand_year_2050.columns = ['End-use category', 'Unit'] + [f'{col} 2050' for col in end_uses_demand_year_2050.columns[2:]]
end_uses_demand_year_combined = pd.merge(end_uses_demand_year_2023, end_uses_demand_year_2050, on=['End-use category', 'Unit'], how='outer').fillna(0)

In [ ]:
# Order the columns as follows: 
col = [
    'End-use category',
    'Unit',
    'Households 2023',
    'Households 2050',
    'Services 2023',
    'Services 2050',
    'Industry 2023',
    'Industry 2050',
    'Transportation 2023',
    'Transportation 2050',
    'Agriculture 2023',
    'Agriculture 2050',
    'Export 2023',
    'Export 2050',
]
end_uses_demand_year_combined = end_uses_demand_year_combined[col]

In [ ]:
end_uses_demand_year_combined.to_csv('../03_Results/Tables/end_uses_demand.csv', index=False)

# Supplementary information tables

In [ ]:
unit_dict = {
    'Total human health': 'DALY/(cap.yr)',
    'Remaining human health': 'DALY/(cap.yr)',
    'Climate change, short term': 't CO2-eq/(cap.yr)',
    'Climate change, short term (abroad)': 't CO2-eq/(cap.yr)',
    'Climate change, short term (territorial)': 't CO2-eq/(cap.yr)',
    'Total ecosystem quality': 'PDF.m2.yr/(cap.yr)',
    'Remaining ecosystem quality': 'PDF.m2.yr/(cap.yr)',
    'Total human health (direct emissions)': 'DALY/(cap.yr)',
    'Total human health (regionalized part)': 'DALY/(cap.yr)',
    'Climate change, short term (direct emissions)': 't CO2-eq/(cap.yr)',
    'Total ecosystem quality (direct emissions)': 'PDF.m2.yr/(cap.yr)',
    'Total ecosystem quality (regionalized part)': 'PDF.m2.yr/(cap.yr)',
}

In [ ]:
def get_sector_unit(tec, sector, phase):

    if 'storage' in tec.lower():
        if phase == 'Construction':
            if 'CO2' in tec:
                return 'kt CO2'
            else:
                return 'GWh'
        else:
            if 'CO2' in tec:
                return 'Mt CO2'
            else:
                return 'TWh'

    if sector == 'Freight mobility':
        if phase == 'Operation':
            return 'Gtkm/year'
        else:
            return 'Mtkm/h'
    elif sector == 'Passenger mobility':
        if phase == 'Operation':
            return 'Gpkm/year'
        else:
            return 'Mpkm/h'
    else:
        if phase in ['Operation', 'Resource']:
            return 'TWh/year'
        else:
            return 'GW'

## Mapping and unit conversion factors

In [ ]:
mapping = pd.read_csv('../01_Notebooks/Data/SI_mapping.csv')
es_tech_df = pd.read_csv('../01_Notebooks/Data/technology_dictionary.csv')

In [ ]:
# Allows to keep formulas in Excel files
from openpyxl import load_workbook
wb = load_workbook(filename='../01_Notebooks/Data/SI_unit_conversion.xlsx')
conversion_factors = pd.DataFrame(wb[wb.sheetnames[0]].values)
new_header = conversion_factors.iloc[0]
conversion_factors = conversion_factors[1:]
conversion_factors.columns = new_header

In [ ]:
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))
es_tech_name_dict = {k: v for k, v in es_tech_name_dict.items()}

In [ ]:
df_mapping = pd.merge(
    mapping.rename(columns={'Product': 'LCI dataset reference product', 'Activity': 'LCI dataset activity'}),
    conversion_factors[['Name', 'Type', 'Value', 'LCA', 'ESM', 'Assumptions & Sources']].rename(columns={'Value': 'Conversion factor', 'LCA': 'Unit LCA', 'ESM': 'Unit ESM'}),
    on=['Name', 'Type'],
    how='inner',
)

## Results for 2023

In [ ]:
df_contrib_imp_cat_tteq = pd.read_csv('../03_Results/Tables/reference/df_contrib_imp_cat_tteq.csv')
df_contrib_imp_cat_tthh = pd.read_csv('../03_Results/Tables/reference/df_contrib_imp_cat_tthh.csv')
df_contrib_sector_tec_tteq = pd.read_csv('../03_Results/Tables/reference/df_contrib_sector_tec_tteq.csv')
df_contrib_sector_tec_tthh = pd.read_csv('../03_Results/Tables/reference/df_contrib_sector_tec_tthh.csv')
df_config_2023 = pd.read_csv('../03_Results/Tables/reference/df_config.csv')

In [ ]:
df_config_2023 = df_config_2023.rename(columns={'Capacity or production': 'Production, capacity or import'})
df_config_2023['Production, capacity or import'] = df_config_2023.apply(lambda x: x['Production, capacity or import']/1e3 if x['Phase'] in ['Operation', 'Resource'] else x['Production, capacity or import'], axis=1) # from GWh to TWh
df_config_2023['Unit'] = df_config_2023.apply(lambda x: get_sector_unit(x['Technology or resource'], x['Sector'], x['Phase']), axis=1)
df_config_2023['Technology or resource'] = df_config_2023['Technology or resource'].replace(es_tech_name_dict)

In [ ]:
df_contrib_imp_cat_tteq = df_contrib_imp_cat_tteq[df_contrib_imp_cat_tteq['Ecosystem quality (biogenic)'] != 0].rename(columns={'Ecosystem quality (biogenic)': 'Ecosystem quality damage (PDF.m2.yr/(cap.yr))'})
df_contrib_imp_cat_tthh = df_contrib_imp_cat_tthh[df_contrib_imp_cat_tthh['Human health (biogenic)'] != 0].rename(columns={'Human health (biogenic)': 'Human health damage (DALY/(cap.yr))'})
df_contrib_imp_cat_tteq['Regionalization level'] = df_contrib_imp_cat_tteq['Regionalization level'].str.replace('+', ' ')
df_contrib_imp_cat_tthh['Regionalization level'] = df_contrib_imp_cat_tthh['Regionalization level'].str.replace('+', ' ')

In [ ]:
df_contrib_sector_tec = pd.concat([df_contrib_sector_tec_tteq, df_contrib_sector_tec_tthh]).drop_duplicates()
df_contrib_sector_tec = df_contrib_sector_tec[(df_contrib_sector_tec.Value != 0) & (~df_contrib_sector_tec.Value.isna())].rename(columns={'Run': 'Regionalization level'})
df_contrib_sector_tec['Unit'] = df_contrib_sector_tec.apply(lambda x: unit_dict[x['Impact category']], axis=1)
df_contrib_sector_tec['Regionalization level'] = df_contrib_sector_tec['Regionalization level'].str.replace('+', ' ')
df_contrib_sector_tec['Technology or resource'] = df_contrib_sector_tec['Technology or resource'].replace(es_tech_name_dict)

## Results for 2050

In [ ]:
df_contrib_sector_tec_2050 = pd.read_csv('../03_Results/Tables/2050/df_total_impact.csv')
df_contrib_imp_cat_tteq_2050 = pd.read_csv('../03_Results/Tables/2050/df_contrib_imp_cat_tteq.csv')
df_contrib_imp_cat_tthh_2050 = pd.read_csv('../03_Results/Tables/2050/df_contrib_imp_cat_tthh.csv')
df_config_2050 = pd.read_csv('../03_Results/Tables/2050/df_config.csv')
df_paral_coord_impacts = pd.read_csv('../03_Results/Tables/2050/df_values_paral_coord_impacts.csv')

In [ ]:
df_contrib_sector_tec_2050 = df_contrib_sector_tec_2050.melt(
    value_vars=[
        'Total ecosystem quality (biogenic)',
        'Total ecosystem quality (regionalized part)',
        'Total human health (biogenic)',
        'Total human health (regionalized part)',
        'Climate change, short term, total',
        'Climate change, short term, total (abroad)',
        'Climate change, short term, total (territorial)',
        'Remaining human health',
        'Remaining ecosystem quality'
    ],
    id_vars=['Run', 'index', 'Sector', 'Phase']
).rename(columns={'index': 'Technology or resource', 'variable': 'Impact category', 'value': 'Value'})
df_contrib_sector_tec_2050['Impact category'] = df_contrib_sector_tec_2050['Impact category'].apply(lambda x: x.replace(' (biogenic)', ''))
df_contrib_sector_tec_2050['Impact category'] = df_contrib_sector_tec_2050['Impact category'].apply(lambda x: x.replace(', total', ''))
df_contrib_sector_tec_2050 = df_contrib_sector_tec_2050[(df_contrib_sector_tec_2050.Value != 0) & (~df_contrib_sector_tec_2050.Value.isna())]
df_contrib_sector_tec_2050['Run'] = df_contrib_sector_tec_2050['Run'].str.replace('+', ' ')
df_contrib_sector_tec_2050 = df_contrib_sector_tec_2050.rename(columns={'Run': 'Prospective-regionalization level'})
df_contrib_sector_tec_2050['Unit'] = df_contrib_sector_tec_2050.apply(lambda x: unit_dict[x['Impact category']], axis=1)

In [ ]:
df_contrib_imp_cat_tteq_2050['Run'] = df_contrib_imp_cat_tteq_2050['Run'].str.replace('+', ' ')
df_contrib_imp_cat_tthh_2050['Run'] = df_contrib_imp_cat_tthh_2050['Run'].str.replace('+', ' ')
df_contrib_imp_cat_tteq_2050 = df_contrib_imp_cat_tteq_2050[df_contrib_imp_cat_tteq_2050['Ecosystem quality (biogenic)'] != 0].rename(columns={
    'Ecosystem quality (biogenic)': 'Ecosystem quality damage (PDF.m2.yr/(cap.yr))',
    'Run': 'Prospective-regionalization level',
    'index': 'Technology or resource',
})
df_contrib_imp_cat_tthh_2050 = df_contrib_imp_cat_tthh_2050[df_contrib_imp_cat_tthh_2050['Human health (biogenic)'] != 0].rename(columns={
    'Human health (biogenic)': 'Human health damage (DALY/(cap.yr))',
    'Run': 'Prospective-regionalization level',
    'index': 'Technology or resource',
})
df_config_2050 = df_config_2050.rename(columns={'Run': 'Prospective-regionalization level'})
df_config_2050['Unit'] = df_config_2050['Unit'].str.replace('<sub>2</sub>', '2')
df_contrib_imp_cat_tteq_2050['Technology or resource'] = df_contrib_imp_cat_tteq_2050['Technology or resource'].replace(es_tech_name_dict)
df_contrib_imp_cat_tthh_2050['Technology or resource'] = df_contrib_imp_cat_tthh_2050['Technology or resource'].replace(es_tech_name_dict)
df_contrib_sector_tec_2050['Technology or resource'] = df_contrib_sector_tec_2050['Technology or resource'].replace(es_tech_name_dict)

## Sensitivity analysis

In [ ]:
df_sensi = pd.read_csv('../03_Results/Tables/sensitivity_analysis/df_total_impact.csv')
df_cost = pd.read_csv('../03_Results/Tables/sensitivity_analysis/df_total_cost.csv')

In [ ]:
df_sensi = aggregate_mobility_submodels(df_sensi)

In [ ]:
df_sensi = df_sensi[df_sensi['index'] != 'CO2_E'].sort_values('Run')
df_sensi['index'] = df_sensi['index'].apply(lambda x: es_tech_name_dict[x])
df_sensi['Regionalization level'] = df_sensi['Regionalization level'].apply(lambda x: reg_level_name_dict_2050[x].replace('+', ' '))
df_cost['Regionalization level'] = df_cost['Regionalization level'].apply(lambda x: reg_level_name_dict_2050[x].replace('+', ' '))
df_sensi = df_sensi.rename(columns={'Run': 'CCS availability (Mt CO2)', 'index': 'Technology or resource'})
df_cost['TotalCost'] *= 1e-3  # from MCAD to BCAD
df_cost = df_cost.rename(columns={'Run': 'CCS availability (Mt CO2)', 'TotalCost': 'System total cost (BCAD)'})
df_sensi.columns = df_sensi.columns.str.replace(' (biogenic)', '')
df_sensi.columns = df_sensi.columns.str.replace(', total', '')
df_sensi.columns = [f"{i} ({unit_dict[i]})" if i in unit_dict else i for i in df_sensi.columns]
df_sensi['Prospective-regionalization level'] = df_sensi.apply(lambda x: f"{x['Regionalization level']}  {x['SSP-RCP']}", axis=1)
df_cost['Prospective-regionalization level'] = df_cost.apply(lambda x: f"{x['Regionalization level']}  {x['SSP-RCP']}", axis=1)
df_sensi = df_sensi.drop(columns=['Regionalization level', 'SSP-RCP'])
df_cost = df_cost.drop(columns=['Regionalization level', 'SSP-RCP', 'Configuration', 'min_TotalCost'])

In [ ]:
df_sensi_impact = df_sensi[[
    'Technology or resource',
    'Phase',
    'Sector',
    'CCS availability (Mt CO2)',
    'Prospective-regionalization level',
    'Climate change, short term (t CO2-eq/(cap.yr))',
    'Total human health (DALY/(cap.yr))',
    'Total ecosystem quality (PDF.m2.yr/(cap.yr))',
    'Remaining human health (DALY/(cap.yr))',
    'Remaining ecosystem quality (PDF.m2.yr/(cap.yr))',
]]

df_sensi_config = df_sensi[[
    'Technology or resource',
    'Phase',
    'Sector',
    'CCS availability (Mt CO2)',
    'Prospective-regionalization level',
    'Production, capacity or import',
]]

In [ ]:
df_sensi_impact = df_sensi_impact[df_sensi_impact['Total human health (DALY/(cap.yr))'] != 0]
df_sensi_config = df_sensi_config[df_sensi_config['Production, capacity or import'] != 0]

In [ ]:
df_sensi_config['Unit'] = df_sensi_config.apply(lambda x: get_sector_unit(x['Technology or resource'], x['Sector'], x['Phase']), axis=1)

## Concatenate in one Excel file

In [63]:
readme = [
    ['', '', ''],
    ['Sheet in the present Supplementary Information S2',	'Corresponding Figure(s) in the main manuscript', 'Presented data'],
    ['Table S1', '', 'Mapping and unit conversion factors between EnergyScope technologies/resources and LCI datasets'],
    ['Sources', '', 'List of sources used in Table S1'],
    ['Table S2', '', 'Energy system configuration for the year 2023'],
    ['Table S3', 'Figure 2a', 'Contribution of impact categories to the damage on human health for the year 2023'],
    ['Table S4', 'Figure 2b', 'Contribution of impact categories to the damage on ecosystem quality for the year 2023'],
    ['Table S5', '', 'Contributions of sectors, technologies, and resources to the different impact categories for the year 2023'],
    ['Table S6', 'Figure 4a', 'Contribution of impact categories to the damage on human health for the year 2050'],
    ['Table S7', 'Figure 4b', 'Contribution of impact categories to the damage on ecosystem quality for the year 2050'],
    ['Table S8', 'Figure 5', 'Contributions of sectors, technologies, and resources to the different impact categories for the year 2050'],
    ['Table S9', 'Figures 3b and 6', 'Energy system configurations for the year 2050'],
    ['Table S10', 'Figure 3a', 'Environmental assessment of energy system configuration using the "Spat. Fore. Back." regionalization level, for both background scenarios'],
    ['Table S11', 'Figures 7a, 7b, 7c', 'Energy system configurations obtained during the sensitivity analysis on the availability of carbon capture and storage'],
    ['Table S12', 'Figure 7d', 'Energy system total cost obtained during the sensitivity analysis on the availability of carbon capture and storage'],
    ['Table S13', '', 'Life-cycle impacts obtained during the sensitivity analysis on the availability of carbon capture and storage'],
]

df_readme = pd.DataFrame(readme, columns=['This Supplementary Information provides the underlying data used in the figures of the main manuscript and the Supplementary Information S3', '', ''])

In [64]:
sources = [
    'Althaus, Hans-Joerg, Mike Chudacoff, Roland Hischier, Niels Jungbluth, Maggie Osses, and Alex Primas. 2007. Life Cycle Inventories of Chemicals. Ecoinvent Report No. 8, v2.0. Dübendorf, CH: Swiss Centre for Life Cycle Inventories.',
    'Biedermann, Ferenc. 2023. Comportement de La Population En Matière de Mobilité. Résultats Du Microrecensement Mobilité et Transports 2021. Office fédérale de la statistique.',
    'Boehm, Randall C., Zhibin Yang, David C. Bell, John Feldhausen, and Joshua S. Heyne. 2022. “Lower Heating Value of Jet Fuel from Hydrocarbon Class Concentration Data and Thermo-Chemical Reference Data: An Uncertainty Quantification.” Fuel 311:122542. doi:10.1016/j.fuel.2021.122542.',
    'Chaire Mobilité. 2018. Rapport d’activités 2014-2015, Version Préliminaire. Polytechnique Montréal.',
    'CIRAIG. 2022. Analyse Du Cycle De Vie De Filières Énergétiques Et De Leur Utilisation Pour Le Transport Routier Au Québec – Partie 2: Utilisation.',
    'Engineering ToolBox. 2003. “Fuels - Higher and Lower Calorific Values.” https://www.engineeringtoolbox.com/fuels-higher-calorific-values-d_169.html.',
    'Engineering ToolBox. 2017. “Combustion Heat.” https://www.engineeringtoolbox.com/standard-heat-of-combustion-energy-content-d_1987.html.',
    'Engineering ToolBox. 2018. “Carbon Dioxide - Density and Specific Weight vs. Temperature and Pressure.” https://www.engineeringtoolbox.com/carbon-dioxide-density-specific-weight-temperature-pressure-d_2018.html.',
    'Fischer, Loïc. 2023. “Own Computation from Various Sources” [Excel File]. (Available upon request).',
    'Gerloff, Niklas. 2021. “Comparative Life-Cycle-Assessment Analysis of Three Major Water Electrolysis Technologies While Applying Various Energy Scenarios for a Greener Hydrogen Production.” Journal of Energy Storage 43:102759. doi:10.1016/j.est.2021.102759.',
    'Grasby, S. E., D. M. Allen, S. Bell, Z. Chen, G. Ferguson, A. Jessop, M. Kelman, M. Ko, J. Majorowicz, M. Moore, J. Raymond, and R. Therrien. 2012. Geothermal Energy Resource Potential of Canada. rev. 6914. doi:10.4095/291488.',
    'Hydro-Québec, 2023. Nos Grands Equipements. https://www.hydroquebec.com/data/enseignants/pdf/grands-equipements-hydro-quebec.pdf',
    'IEA-SHC. 2004. “Solar Thermal Statistics Conversion Method - M2 to GWth.”',
    'Kamidelivand, Mitra, Peter Deeney, Fiona Devoy McAuliffe, Kevin Leyne, Michael Togneri, and Jimmy Murphy. 2023. “Scenario Analysis of Cost-Effectiveness of Maintenance Strategies for Fixed Tidal Stream Turbines in the Atlantic Ocean.” Journal of Marine Science and Engineering 11(5):1046. doi:10.3390/jmse11051046.',
    'Limpens, Gauthier. 2021. “Generating Energy Transition Pathways : Application to Belgium.” PhD thesis, Université catholique de Louvain.',
    'Marin-Batista, J. D., J. A. Villamil, S. V. Qaramaleki, C. J. Coronella, A. F. Mohedano, and M. A. De La Rubia. 2020. “Energy Valorization of Cow Manure by Hydrothermal Carbonization and Anaerobic Digestion.” Renewable Energy 160:623–32. doi:10.1016/j.renene.2020.07.003.',
    'Moret, Stefano. 2017. “Strategic Energy Planning under Uncertainty.” PhD thesis, EPFL.',
    'Notter, Dominic A., Marcel Gauch, Rolf Widmer, Patrick Wäger, Anna Stamp, Rainer Zah, and Hans-Jörg Althaus. 2010. “Contribution of Li-Ion Batteries to the Environmental Impact of Electric Vehicles.” Environmental Science & Technology 44(17):6550–56. doi:10.1021/es903729a.',
    'Pöschl, Martina, Shane Ward, and Philip Owende. 2010. “Evaluation of Energy Efficiency of Various Biogas Production and Utilization Pathways.” Applied Energy 87(11):3305–21. doi:10/fwz87p.',
    'Sacchi, R., T. Terlouw, K. Siala, A. Dirnaichner, C. Bauer, B. Cox, C. Mutel, V. Daioglou, and G. Luderer. 2022. “PRospective EnvironMental Impact asSEment (Premise): A Streamlined Approach to Producing Databases for Prospective Life Cycle Assessment Using Integrated Assessment Models.” Renewable and Sustainable Energy Reviews 160:112311. doi:10.1016/j.rser.2022.112311.',
    'Sacchi, Romain, Christian Bauer, and Brian L. Cox. 2021. “Does Size Matter? The Influence of Size, Load Factor, Range Autonomy, and Application Type on the Life Cycle Assessment of Current and Future Medium- and Heavy-Duty Vehicles.” Environmental Science & Technology 55(8):5224–35. doi:10.1021/acs.est.0c07773.',
    'Schnidrig, Jonas, Rachid Cherkaoui, Yasmine Calisesi, Manuele Margni, and François Maréchal. 2023. “On the Role of Energy Infrastructure in the Energy Transition. Case Study of an Energy Independent and CO2 Neutral Energy System for Switzerland.” Frontiers in Energy Research 11:1164813. doi:10.3389/fenrg.2023.1164813.',
    'Schnidrig, Jonas, Tuong-Van Nguyen, Paul Stadler, and François Maréchal. 2020. “Assessment of Green Mobility Scenarios on European Energy Systems.” Master thesis, EPFL.',
    'Spielmann, M., R. Dones, and C. Bauer. 2007. Life Cycle Inventories of Transport Services. Final Report Ecoinvent v2.0 No. 14. Dübendorf, CH: Swiss Centre for Life Cycle Inventories.',
    'The World Bank. 2009. Air Freight: A Market Study with Implications for Landlocked Countries. Washington, D.C.: The International Bank for Reconstruction and Development / The World Bank. https://documents1.worldbank.org/curated/en/265051468324548129/pdf/517470NWP0tp1210Box342045B01PUBLIC1.pdf.',
    'Thomson, R. Camilla, John P. Chick, and Gareth P. Harrison. 2019. “An LCA of the Pelamis Wave Energy Converter.” The International Journal of Life Cycle Assessment 24(1):51–63. doi:10.1007/s11367-018-1504-2.',
    'Van Der Giesen, Coen, René Kleijn, and Gert Jan Kramer. 2014. “Energy and Climate Impacts of Producing Synthetic Hydrocarbon Fuels from CO 2.” Environmental Science & Technology 48(12):7111–21. doi:10.1021/es500191g.',
    'Wernet, Gregor, Christian Bauer, Bernhard Steubing, Jürgen Reinhard, Emilia Moreno-Ruiz, and Bo Weidema. 2016. “The Ecoinvent Database Version 3 (Part I): Overview and Methodology.” The International Journal of Life Cycle Assessment 21(9):1218–30. doi:10.1007/s11367-016-1087-8.',
    'Wikipedia. 2023. “Heat of Combustion.” https://en.wikipedia.org/wiki/Heat_of_combustion.',
    'Willauer, Heather, Dennis Hardy, Kenneth Schultz, and Frederick Williams. 2012. “The Feasibility and Current Estimated Capital Costs of Producing Jet Fuel at Sea Using Carbon Dioxide and Hydrogen.” Journal of Renewable Sustainable Energy. doi:10.1063/1.4719723.',
    'World Nuclear Association. n.d. “Heat Values of Various Fuels.” Retrieved April 16, 2024. https://world-nuclear.org/information-library/facts-and-figures/heat-values-of-various-fuels.aspx.',
    'WSP, 2021. Inventaire de la biomasse disponible pour produire de la bioénergie et portrait de la production de la bioénergie sur le territoire québécois (No. Réf. WSP : 201-03354-00). Rapport réalisé par WSP Canada Inc., pour le compte du Ministère de l’Énergie et des Ressources naturelles.',
]

df_sources = pd.DataFrame(sources, columns=['References'])

In [65]:
with pd.ExcelWriter('../03_Results/Tables/SI_results_figures.xlsx', engine='xlsxwriter') as writer:
    workbook = writer.book

    # Mapping and conversion factors
    name = 'README'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_readme.to_excel(writer, sheet_name=name, index=False, startrow=0, startcol=0)

    # Mapping and conversion factors
    name = 'Table S1'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_mapping.to_excel(writer, sheet_name=name, index=False)

    # Sources for conversion factors assumptions
    name = 'Sources'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_sources.to_excel(writer, sheet_name=name, index=False)

    # Configuration for 2023
    name = 'Table S2'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_config_2023.to_excel(writer, sheet_name=name, index=False)

    # Contributions of impact categories on TTHH for 2023
    name = 'Table S3'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_imp_cat_tthh.to_excel(writer, sheet_name=name, index=False)

    # Contributions of impact categories on TTEQ for 2023
    name = 'Table S4'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_imp_cat_tteq.to_excel(writer, sheet_name=name, index=False)

    # Contributions of technologies and sectors for 2023
    name = 'Table S5'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_sector_tec.to_excel(writer, sheet_name=name, index=False)

    # Contributions of impact categories on TTHH for 2050
    name = 'Table S6'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_imp_cat_tthh_2050.to_excel(writer, sheet_name=name, index=False)

    # Contributions of impact categories on TTEQ for 2050
    name = 'Table S7'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_imp_cat_tteq_2050.to_excel(writer, sheet_name=name, index=False)

    # Contributions of technologies and sectors for 2050
    name = 'Table S8'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_contrib_sector_tec_2050.to_excel(writer, sheet_name=name, index=False)

    # Configuration per modeling level for 2050
    name = 'Table S9'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_config_2050.to_excel(writer, sheet_name=name, index=False)

    # Parallel coordinates plot for impacts
    name = 'Table S10'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_paral_coord_impacts.to_excel(writer, sheet_name=name, index=False)

    # Sensitivity analysis - configurations
    name = 'Table S11'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_sensi_config.to_excel(writer, sheet_name=name, index=False)

    # Sensitivity analysis - total costs
    name = 'Table S12'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_cost.to_excel(writer, sheet_name=name, index=False)

    # Sensitivity analysis - environmental impacts
    name = 'Table S13'
    writer.sheets[name] = workbook.add_worksheet(name)
    df_sensi_impact.to_excel(writer, sheet_name=name, index=False)